# 07 — Model Dataset Creation

**Phase 12 + Phase 13.** Build the final `community_area × date` modeling
matrix: assemble features (calendar, weather, historical crime), create
the `high_crime_day` target, audit for leakage, and apply the chronological
train/validation/test split. No model is trained here — this notebook's
only job is to produce a clean, leakage-safe `modeling_dataset.parquet`
for notebook 08.

**Input.** `data/interim/crime_weather_daily.parquet` (notebook 06 output).

**Outputs.** `data/processed/modeling_dataset.parquet`,
`data/processed/feature_metadata.csv`.

**Non-negotiable rules enforced in this notebook**
1. `high_crime_day` threshold computed from the **training period only**
   (blueprint Phase 12) — never from validation/test data.
2. Same-day crime outcome columns (`crime_count`, `violent_crime_count`,
   `property_crime_count`, `drug_crime_count`, `arrest_rate`,
   `domestic_rate`) are **excluded from the feature matrix** — they are
   the basis of the target itself, not legitimate predictors.
3. Split is **strictly chronological** (train 2021-2022 / val 2023 /
   test 2024-2025, per `config.py`) — never a random split.
4. No auto-imputation — missing values from lag/rolling warm-up are
   dropped and counted, not filled in.


## 02 — Environment & Imports

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

from src import config
from src.features.temporal_features import create_date_features
from src.features.crime_lags import create_crime_lag_features, LAG_DAYS, ROLLING_WINDOWS

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 130)

RANDOM_STATE = config.RANDOM_STATE
TARGET_PERCENTILE = config.TARGET_PERCENTILE

MERGED_PATH = config.DATA_INTERIM / "crime_weather_daily.parquet"
OUT_DATASET_PATH = config.DATA_PROCESSED / "modeling_dataset.parquet"
OUT_METADATA_PATH = config.DATA_PROCESSED / "feature_metadata.csv"
config.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

config.TRAIN_YEARS, config.VALIDATION_YEARS, config.TEST_YEARS, TARGET_PERCENTILE

((2021, 2022), (2023,), (2024, 2025), 0.75)

## 03 — Load & Validate

Fails fast if notebook 06's output is missing — no silent fallback.

In [2]:
if not MERGED_PATH.exists():
    raise FileNotFoundError(f"{MERGED_PATH} not found. Run 06_crime_weather_analysis.ipynb first.")

df = pd.read_parquet(MERGED_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["community_area", "date"]).reset_index(drop=True)
print(f"loaded: {df.shape}")
df.head()

loaded: (140602, 35)


,date,crime_count,violent_crime_count,property_crime_count,drug_crime_count,unique_crime_types,arrest_rate,domestic_rate,community_area,crime_count_lag_1,crime_count_lag_3,crime_count_lag_7,crime_count_lag_14,crime_count_lag_28,crime_rolling_7,crime_rolling_14,crime_rolling_28,recent_7d_avg,previous_7d_avg,trend_ratio,violent_ratio,property_ratio,AWND,SNOW,TMAX,TMIN,PRCP,avg_temp,temp_range,is_rain,is_snow,is_heavy_rain,is_heavy_snow,is_extreme_heat,is_extreme_cold
0,2021-01-01,16,1,9,2,8,0.187500,0.187500,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.86,1.3,35,19,0.38,27.0,16,True,True,False,False,False,True
1,2021-01-02,6,2,4,0,5,0.000000,0.166667,1,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.26,0.0,35,28,0.00,31.5,7,False,False,False,False,False,False
2,2021-01-03,10,4,5,0,6,0.100000,0.300000,1,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.93,0.4,33,26,0.03,29.5,7,True,True,False,False,False,False
3,2021-01-04,7,1,4,1,5,0.285714,0.142857,1,10.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.83,0.0,30,24,0.00,27.0,6,False,False,False,False,False,False
4,2021-01-05,11,2,8,0,4,0.000000,0.272727,1,7.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.59,0.0,37,29,0.00,33.0,8,False,False,False,False,False,False


In [3]:
REQUIRED_BASE_COLS = ["community_area", "date", "crime_count", "violent_crime_count",
                      "property_crime_count", "drug_crime_count", "arrest_rate", "domestic_rate",
                      "avg_temp", "temp_range", "PRCP", "SNOW", "AWND",
                      "is_rain", "is_snow", "is_heavy_rain", "is_heavy_snow",
                      "is_extreme_heat", "is_extreme_cold"]
missing = [c for c in REQUIRED_BASE_COLS if c not in df.columns]
if missing:
    raise KeyError(f"missing required columns from crime_weather_daily.parquet: {missing}")

assert df["date"].min() == pd.Timestamp(config.START_DATE)
assert df["date"].max() == pd.Timestamp(config.END_DATE)
assert (df["crime_count"] >= 0).all(), "negative crime_count is impossible"
n_areas = df["community_area"].nunique()
n_dates = df["date"].nunique()
assert len(df) == n_areas * n_dates, "not a full community_area x date grid"
print(f"Validated: {n_areas} areas x {n_dates} dates = {len(df)} rows, full grid, in-scope dates, required columns present.")

Validated: 77 areas x 1826 dates = 140602 rows, full grid, in-scope dates, required columns present.


## 04 — Calendar Features

Reused from `features.temporal_features.create_date_features` (already
built and used in notebooks 05/06) — not reimplemented here.

In [4]:
df = create_date_features(df, date_col="date")
new_calendar_cols = ["year", "month", "day", "day_of_week", "week_of_year", "quarter", "season", "is_weekend"]
assert all(c in df.columns for c in new_calendar_cols)
df[["community_area", "date"] + new_calendar_cols].head()

,community_area,date,year,month,day,day_of_week,week_of_year,quarter,season,is_weekend
0,1,2021-01-01,2021,1,1,4,53,1,Winter,False
1,1,2021-01-02,2021,1,2,5,53,1,Winter,True
2,1,2021-01-03,2021,1,3,6,53,1,Winter,True
3,1,2021-01-04,2021,1,4,0,1,1,Winter,False
4,1,2021-01-05,2021,1,5,1,1,1,Winter,False


## 05 — Historical Crime (Lag/Rolling) Features

`crime_daily.parquet` (built in notebook 03) is expected to already carry
these — Phase 11 is implemented there, not duplicated here. This section
**checks** for that rather than assuming it, and only computes them if
genuinely absent, so the notebook is correct either way.

In [5]:
LAG_COLS = [f"crime_count_lag_{d}" for d in LAG_DAYS]
ROLLING_COLS = [f"crime_rolling_{w}" for w in ROLLING_WINDOWS]
DERIVED_HIST_COLS = ["recent_7d_avg", "previous_7d_avg", "trend_ratio", "violent_ratio", "property_ratio"]
ALL_HIST_COLS = LAG_COLS + ROLLING_COLS + DERIVED_HIST_COLS

already_present = [c for c in ALL_HIST_COLS if c in df.columns]
missing_hist = [c for c in ALL_HIST_COLS if c not in df.columns]
print(f"already present: {len(already_present)}/{len(ALL_HIST_COLS)}")
if missing_hist:
    print(f"computing missing: {missing_hist}")
    df = create_crime_lag_features(df)
else:
    print("all historical crime features already present -- not recomputed.")
assert all(c in df.columns for c in ALL_HIST_COLS)

already present: 13/13
all historical crime features already present -- not recomputed.


### Leakage-safety spot-check

`crime_count_lag_1` for a given `community_area`/`date` must equal that
area's `crime_count` from exactly one day earlier — verified directly
against the raw series, not assumed from the column name.

In [6]:
check = df[["community_area", "date", "crime_count", "crime_count_lag_1"]].copy()
check["expected_lag_1"] = check.groupby("community_area")["crime_count"].shift(1)

comparable = check.dropna(subset=["crime_count_lag_1", "expected_lag_1"])
mismatches = comparable[comparable["crime_count_lag_1"] != comparable["expected_lag_1"]]
print(f"{len(comparable)} rows compared, {len(mismatches)} mismatches")
assert len(mismatches) == 0, "crime_count_lag_1 does not match a true 1-day shift -- leakage or bug, stop here"
print("PASS: crime_count_lag_1 is a true prior-day value for every checkable row.")

140525 rows compared, 0 mismatches
PASS: crime_count_lag_1 is a true prior-day value for every checkable row.


In [7]:
# Second check: crime_rolling_7 must never include the current day's own crime_count.
# Reconstruct independently (shift(1) then rolling(7).sum()) and compare.
recon = df.groupby("community_area")["crime_count"].apply(lambda s: s.shift(1).rolling(7).sum())
recon = recon.reset_index(level=0, drop=True)
comparable_roll = df["crime_rolling_7"].dropna()
mismatches_roll = (recon.loc[comparable_roll.index] != comparable_roll).sum()
print(f"crime_rolling_7 mismatches vs independent reconstruction: {mismatches_roll}")
assert mismatches_roll == 0, "crime_rolling_7 includes same-day information or is otherwise wrong -- stop here"
print("PASS: crime_rolling_7 is built from strictly prior days.")

crime_rolling_7 mismatches vs independent reconstruction: 0
PASS: crime_rolling_7 is built from strictly prior days.


In [8]:
n_missing_per_hist_col = df[ALL_HIST_COLS].isna().sum()
df["_year_tmp"] = df["date"].dt.year
years_with_na = {c: sorted(df.loc[df[c].isna(), "_year_tmp"].unique().tolist()) for c in ALL_HIST_COLS if df[c].isna().any()}
df = df.drop(columns="_year_tmp")
n_missing_per_hist_col

crime_count_lag_1       77
crime_count_lag_3      231
crime_count_lag_7      539
crime_count_lag_14    1078
crime_count_lag_28    2156
crime_rolling_7        539
crime_rolling_14      1078
crime_rolling_28      2156
recent_7d_avg          539
previous_7d_avg       1078
trend_ratio           1119
violent_ratio          580
property_ratio         580
dtype: int64

In [9]:
print("Years affected by NaN, per column (lag/rolling should show 2021 only; ratios may show all years):")
for col, years in years_with_na.items():
    print(f"  {col}: {years}")

Years affected by NaN, per column (lag/rolling should show 2021 only; ratios may show all years):
  crime_count_lag_1: [2021]
  crime_count_lag_3: [2021]
  crime_count_lag_7: [2021]
  crime_count_lag_14: [2021]
  crime_count_lag_28: [2021]
  crime_rolling_7: [2021]
  crime_rolling_14: [2021]
  crime_rolling_28: [2021]
  recent_7d_avg: [2021]
  previous_7d_avg: [2021]
  trend_ratio: [2021, 2022, 2023, 2024, 2025]
  violent_ratio: [2021, 2022, 2023, 2024, 2025]
  property_ratio: [2021, 2022, 2023, 2024, 2025]


**Two different mechanisms — not one.** `crime_count_lag_*` and
`crime_rolling_*` are NaN only during each area's warm-up window (first
~28 days, all in 2021) — expected, structural, and harmless.

`trend_ratio`, `violent_ratio`, `property_ratio` are different:
`features.crime_lags` divides by a trailing sum that it explicitly
replaces with `NaN` when it is zero (`.replace(0, pd.NA)`, see the
module's source), to avoid a division-by-zero. A trailing 7-day window
with **zero crime** is not a 2021-only warm-up event — it can happen in
any year, for any lower-crime area, any time. If the years-affected list
above shows these three columns spanning years beyond 2021, dropping
their NaN rows in Section 08 would remove **validation/test** rows too,
non-randomly (skewed toward lower-crime area-days) — a real evaluation-set
bias risk, not a warm-up technicality. Resolved by excluding these three
columns from the model's feature set in Section 06, not by imputing them.

## 06 — Feature Inventory & Same-Day Leakage Classification

Every column is classified before it goes anywhere near a feature matrix
— three groups, each with a distinct reason:

- **`LEAKY_SAME_DAY_COLS`** — the target's own raw material. Excluded
  unconditionally.
- **`REDUNDANT_COLS`** — not leaky, but collinear with a feature already
  included (`TMAX`/`TMIN` vs. `avg_temp`, r=0.95-0.99 per notebook 05).
  Excluded for redundancy, kept in the source data for transparency.
- **`STRUCTURAL_NA_RISK_COLS`** — not leaky, not redundant, but NaN
  beyond warm-up (Section 05) in a way that would bias which val/test
  rows survive Section 08's drop. Excluded to protect evaluation-set
  integrity.
- **`FEATURE_COLS`** — everything that goes into `X`.

In [10]:
LEAKY_SAME_DAY_COLS = ["crime_count", "violent_crime_count", "property_crime_count",
                       "drug_crime_count", "arrest_rate", "domestic_rate", "unique_crime_types"]
LEAKY_SAME_DAY_COLS = [c for c in LEAKY_SAME_DAY_COLS if c in df.columns]

REDUNDANT_COLS = ["TMAX", "TMIN"]  # collinear with avg_temp by construction -- see notebook 05
REDUNDANT_COLS = [c for c in REDUNDANT_COLS if c in df.columns]

# Excluded per Section 05's evidence: NaN via crime_lags.py's division-by-zero
# guard (.replace(0, pd.NA)), not confined to warm-up -- would non-randomly
# bias val/test row survival in Section 08 if kept as model features.
STRUCTURAL_NA_RISK_COLS = ["trend_ratio", "violent_ratio", "property_ratio"]
STRUCTURAL_NA_RISK_COLS = [c for c in STRUCTURAL_NA_RISK_COLS if c in df.columns]

CALENDAR_FEATURE_COLS = ["year", "month", "day_of_week", "week_of_year", "quarter", "season", "is_weekend"]
WEATHER_FEATURE_COLS = ["avg_temp", "temp_range", "PRCP", "SNOW", "AWND",
                        "is_rain", "is_snow", "is_heavy_rain", "is_heavy_snow",
                        "is_extreme_heat", "is_extreme_cold"]
HISTORICAL_FEATURE_COLS = [c for c in ALL_HIST_COLS if c not in STRUCTURAL_NA_RISK_COLS]
ID_COLS = ["community_area", "date"]

FEATURE_COLS = CALENDAR_FEATURE_COLS + WEATHER_FEATURE_COLS + HISTORICAL_FEATURE_COLS

excluded_all = set(LEAKY_SAME_DAY_COLS) | set(REDUNDANT_COLS) | set(STRUCTURAL_NA_RISK_COLS)
overlap = set(FEATURE_COLS) & excluded_all
assert not overlap, f"excluded column(s) present in FEATURE_COLS: {overlap}"

accounted = set(ID_COLS) | set(FEATURE_COLS) | excluded_all | {"day"}
unclassified = [c for c in df.columns if c not in accounted]
print(f"features: {len(FEATURE_COLS)}   same-day (excluded): {len(LEAKY_SAME_DAY_COLS)}   "
      f"redundant (excluded): {len(REDUNDANT_COLS)}   structural-NA-risk (excluded): {len(STRUCTURAL_NA_RISK_COLS)}   "
      f"id: {len(ID_COLS)}")
print(f"unclassified columns (review before proceeding): {unclassified}")

features: 28   same-day (excluded): 7   redundant (excluded): 2   structural-NA-risk (excluded): 3   id: 2
unclassified columns (review before proceeding): []


In [11]:
assert len(unclassified) == 0, (
    f"{len(unclassified)} column(s) not classified as feature/leaky/id -- "
    f"classify explicitly before building X, do not silently include or drop: {unclassified}"
)
print("Every column accounted for. Feature matrix will use only FEATURE_COLS.")

Every column accounted for. Feature matrix will use only FEATURE_COLS.


## 07 — Target Creation: `high_crime_day`

**Rule (blueprint Phase 12).** `high_crime_day = 1` if a community area's
`crime_count` on a date exceeds *that area's own* historical
75th percentile, else 0. The percentile is
computed **once, from the training period only**
(`2021-2022`), then applied unchanged to
every row — train, validation, and test alike. This is a fitted parameter,
not a per-split recomputation; recomputing it on validation/test would be
leakage.

In [12]:
train_mask = df["year"].isin(config.TRAIN_YEARS)
print(f"training rows for threshold fitting: {train_mask.sum()} ({train_mask.mean():.1%} of data)")

area_thresholds = (
    df.loc[train_mask]
    .groupby("community_area")["crime_count"]
    .quantile(TARGET_PERCENTILE)
    .rename("crime_count_threshold")
)
print(f"{len(area_thresholds)} area-level thresholds computed from training data only.")
area_thresholds.describe()

training rows for threshold fitting: 56210 (40.0% of data)
77 area-level thresholds computed from training data only.


count    77.000000
mean      9.928571
std       7.493183
min       1.000000
25%       4.000000
50%       7.000000
75%      13.000000
max      38.000000
Name: crime_count_threshold, dtype: float64

In [13]:
assert area_thresholds.notna().all(), "at least one area has an undefined training-period threshold -- check for areas with all-NaN/zero training crime_count"

df = df.merge(area_thresholds, on="community_area", how="left")
df["high_crime_day"] = (df["crime_count"] > df["crime_count_threshold"]).astype(int)

overall_rate = df["high_crime_day"].mean()
print(f"overall high_crime_day rate: {overall_rate:.1%} (target is ~25% by construction, since threshold = 75th percentile of the TRAINING distribution only -- expect drift in val/test if crime patterns shifted)")
df[["community_area", "date", "crime_count", "crime_count_threshold", "high_crime_day"]].head()

overall high_crime_day rate: 25.8% (target is ~25% by construction, since threshold = 75th percentile of the TRAINING distribution only -- expect drift in val/test if crime patterns shifted)


,community_area,date,crime_count,crime_count_threshold,high_crime_day
0,1,2021-01-01,16,12.0,1
1,1,2021-01-02,6,12.0,0
2,1,2021-01-03,10,12.0,0
3,1,2021-01-04,7,12.0,0
4,1,2021-01-05,11,12.0,0


In [14]:
rate_by_split_year = df.groupby("year")["high_crime_day"].mean()
rate_by_split_year

year
2021    0.156663
2022    0.252019
2023    0.324462
2024    0.311333
2025    0.244939
Name: high_crime_day, dtype: float64

**Reading.** Overall rate: 25.8% (close to 25% by construction — good).
By year, the rate is **not stable**: 2021 15.7% → 2022 25.2% → 2023
**32.4%** → 2024 31.1% → 2025 24.5%.

Crime rose relative to the 2021-2022 training baseline through 2023-2024,
then eased back toward baseline in 2025. This is a real drift, not noise —
directly matches the blueprint's stated risk ("crime patterns drift over
time → model stability risk"). Notebook 08 will see this as a validation
set (2023) with a *harder* class balance than training, and should not
be surprised by it.

## 08 — Missing-Value Audit

Lag/rolling warm-up rows (Section 05) have no valid feature values yet —
dropped from the modeling dataset, not imputed (blueprint rule: no
auto-imputation). Counted explicitly before and after.

In [15]:
required_for_modeling = FEATURE_COLS + ["high_crime_day"]
rows_with_na = df[required_for_modeling].isna().any(axis=1)
n_before = len(df)
n_dropped = int(rows_with_na.sum())

print(f"rows before drop : {n_before}")
print(f"rows with NaN in a required column: {n_dropped} ({n_dropped / n_before:.2%})")

na_by_col = df[required_for_modeling].isna().sum()
na_by_col[na_by_col > 0]

rows before drop : 140602
rows with NaN in a required column: 2156 (1.53%)


crime_count_lag_1       77
crime_count_lag_3      231
crime_count_lag_7      539
crime_count_lag_14    1078
crime_count_lag_28    2156
crime_rolling_7        539
crime_rolling_14      1078
crime_rolling_28      2156
recent_7d_avg          539
previous_7d_avg       1078
dtype: int64

In [16]:
dropped_rows_preview = df.loc[rows_with_na, ["community_area", "date", "year"]].copy()
dropped_by_year = dropped_rows_preview["year"].value_counts().sort_index()
print(f"dropped rows span {dropped_rows_preview['date'].min().date()} to {dropped_rows_preview['date'].max().date()}")
print("dropped rows by year:")
print(dropped_by_year)

df_clean = df.loc[~rows_with_na].reset_index(drop=True)
print(f"\nrows after drop  : {len(df_clean)} ({len(df_clean) / n_before:.2%} retained)")
assert df_clean[required_for_modeling].isna().sum().sum() == 0
assert df_clean["date"].min() > pd.Timestamp(config.START_DATE), "expected the earliest dates to be trimmed by warm-up drop"


dropped rows span 2021-01-01 to 2021-01-28
dropped rows by year:
year
2021    2156
Name: count, dtype: int64

rows after drop  : 138446 (98.47% retained)


**Reading.** Since Section 06 already excludes `trend_ratio`/
`violent_ratio`/`property_ratio` (the structural-NA-risk columns), the
only remaining NaN source in `required_for_modeling` is true warm-up —
each area's first ~28 days. The year breakdown above should show **2021
only**. If any other year appears, a required column outside
`STRUCTURAL_NA_RISK_COLS` has an unexpected non-warm-up NaN — stop and
investigate before proceeding, do not drop and move on silently.

In [17]:
assert set(dropped_by_year.index) == {2021}, (
    f"drop touched year(s) beyond 2021 warm-up: {sorted(dropped_by_year.index)} -- "
    f"a feature column has non-warm-up missingness that Section 06 did not account for. Stop here."
)
print("Confirmed: warm-up drop is 2021-only. Validation (2023) and test (2024-2025) rows are fully intact.")

Confirmed: warm-up drop is 2021-only. Validation (2023) and test (2024-2025) rows are fully intact.


## 09 — Chronological Split

Per `config.py`: train `(2021, 2022)`, validation
`(2023,)`, test `(2024, 2025)` — assigned by
calendar year, never by a random shuffle (blueprint rule 7).

In [18]:
def assign_split(year):
    if year in config.TRAIN_YEARS:
        return "train"
    if year in config.VALIDATION_YEARS:
        return "validation"
    if year in config.TEST_YEARS:
        return "test"
    raise ValueError(f"year {year} not covered by any configured split -- check config.py")

df_clean["split"] = df_clean["year"].apply(assign_split)
split_counts = df_clean["split"].value_counts()
split_counts

split
test          56287
train         54054
validation    28105
Name: count, dtype: int64

In [19]:
split_date_ranges = df_clean.groupby("split")["date"].agg(["min", "max"])
split_date_ranges = split_date_ranges.loc[["train", "validation", "test"]]
split_date_ranges

,min,max
split,,
train,2021-01-29,2022-12-31
validation,2023-01-01,2023-12-31
test,2024-01-01,2025-12-31


In [20]:
# Strict ordering: every train date < every validation date < every test date. No overlap.
train_max = df_clean.loc[df_clean["split"] == "train", "date"].max()
val_min = df_clean.loc[df_clean["split"] == "validation", "date"].min()
val_max = df_clean.loc[df_clean["split"] == "validation", "date"].max()
test_min = df_clean.loc[df_clean["split"] == "test", "date"].min()

assert train_max < val_min, "train/validation overlap or out of order"
assert val_max < test_min, "validation/test overlap or out of order"
print(f"train ends {train_max.date()}  <  validation starts {val_min.date()}")
print(f"validation ends {val_max.date()}  <  test starts {test_min.date()}")
print("PASS: strictly chronological, non-overlapping split.")

train ends 2022-12-31  <  validation starts 2023-01-01
validation ends 2023-12-31  <  test starts 2024-01-01
PASS: strictly chronological, non-overlapping split.


In [21]:
class_balance_by_split = df_clean.groupby("split")["high_crime_day"].agg(["mean", "count"]).loc[["train", "validation", "test"]]
class_balance_by_split.columns = ["high_crime_day_rate", "n_rows"]
class_balance_by_split

,high_crime_day_rate,n_rows
split,,
train,0.207034,54054
validation,0.324462,28105
test,0.278181,56287


**Reading.** `high_crime_day` rate by split: train 20.7%, **validation
32.4%**, test 27.8%. This mirrors Section 07's year-by-year drift exactly
— validation (2023 alone) lands on the peak drift year, so it is the
hardest, least-balanced split. Confirms the imbalance handling in
notebook 08 (Phase 15) needs to be split-aware, not tuned to training's
20.7% alone.

### Preview: `TimeSeriesSplit` for cross-validation within training (setup for 08)

Not executed here (no model is trained in this notebook) — this only
confirms the training partition is CV-ready for Phase 16's
`RandomizedSearchCV` in notebook 08.

In [22]:
train_dates_sorted = np.sort(df_clean.loc[df_clean["split"] == "train", "date"].unique())
tscv = TimeSeriesSplit(n_splits=3)
for i, (tr_idx, va_idx) in enumerate(tscv.split(train_dates_sorted)):
    print(f"fold {i}: train dates {train_dates_sorted[tr_idx[0]].astype('datetime64[D]')} to "
          f"{train_dates_sorted[tr_idx[-1]].astype('datetime64[D]')}  |  "
          f"val dates {train_dates_sorted[va_idx[0]].astype('datetime64[D]')} to "
          f"{train_dates_sorted[va_idx[-1]].astype('datetime64[D]')}")

fold 0: train dates 2021-01-29 to 2021-07-24  |  val dates 2021-07-25 to 2022-01-15
fold 1: train dates 2021-01-29 to 2022-01-15  |  val dates 2022-01-16 to 2022-07-09
fold 2: train dates 2021-01-29 to 2022-07-09  |  val dates 2022-07-10 to 2022-12-31


## 10 — Leakage Audit (consolidated)

Every check performed in this notebook, gathered into one pass/fail
table — the mandatory Phase 13 deliverable.

In [23]:
audit_checks = []

def record(name, passed, detail=""):
    audit_checks.append({"check": name, "passed": bool(passed), "detail": detail})

record("Target threshold fit on training years only", train_mask.sum() > 0 and (df.loc[train_mask, "year"].isin(config.TRAIN_YEARS)).all(),
       f"{train_mask.sum()} rows, years {sorted(df.loc[train_mask, 'year'].unique())}")
record("No same-day crime column present in FEATURE_COLS", not (set(FEATURE_COLS) & set(LEAKY_SAME_DAY_COLS)),
       f"excluded: {LEAKY_SAME_DAY_COLS}")
record("No redundant column present in FEATURE_COLS", not (set(FEATURE_COLS) & set(REDUNDANT_COLS)),
       f"excluded: {REDUNDANT_COLS}")
record("No structural-NA-risk column present in FEATURE_COLS", not (set(FEATURE_COLS) & set(STRUCTURAL_NA_RISK_COLS)),
       f"excluded: {STRUCTURAL_NA_RISK_COLS}")
record("Every column classified (feature / leaky / redundant / structural-NA-risk / id)", len(unclassified) == 0)
record("Warm-up drop confined to 2021 (no non-random val/test row loss)", set(dropped_by_year.index) == {2021})
record("crime_count_lag_1 verified as true 1-day shift", len(mismatches) == 0, f"{len(comparable)} rows checked")
record("crime_rolling_7 verified to exclude current day", mismatches_roll == 0)
record("No missing values in final feature/target columns", df_clean[required_for_modeling].isna().sum().sum() == 0)
record("Warm-up drop did not touch validation/test dates", df_clean["date"].min() > pd.Timestamp(config.START_DATE))
record("Split is strictly chronological, non-overlapping", train_max < val_min < test_min)
record("Split years match config.py exactly",
       set(df_clean.loc[df_clean['split']=='train','year']) == set(config.TRAIN_YEARS) and
       set(df_clean.loc[df_clean['split']=='validation','year']) == set(config.VALIDATION_YEARS) and
       set(df_clean.loc[df_clean['split']=='test','year']) == set(config.TEST_YEARS))
record("Row count conserved through calendar/target/split steps (only warm-up rows dropped)",
       len(df_clean) == n_before - n_dropped)

leakage_audit_df = pd.DataFrame(audit_checks)
leakage_audit_df

,check,passed,detail
0,Target threshold fit on training years only,True,"56210 rows, years [np.int32(2021), np.int32(20..."
1,No same-day crime column present in FEATURE_COLS,True,"excluded: ['crime_count', 'violent_crime_count..."
2,No redundant column present in FEATURE_COLS,True,"excluded: ['TMAX', 'TMIN']"
3,No structural-NA-risk column present in FEATUR...,True,"excluded: ['trend_ratio', 'violent_ratio', 'pr..."
4,Every column classified (feature / leaky / red...,True,
5,Warm-up drop confined to 2021 (no non-random v...,True,
6,crime_count_lag_1 verified as true 1-day shift,True,140525 rows checked
7,crime_rolling_7 verified to exclude current day,True,
8,No missing values in final feature/target columns,True,
9,Warm-up drop did not touch validation/test dates,True,


In [24]:
assert leakage_audit_df["passed"].all(), (
    f"{(~leakage_audit_df['passed']).sum()} leakage check(s) FAILED -- stop, "
    f"do not save a dataset that fails this audit:\n{leakage_audit_df[~leakage_audit_df['passed']]}"
)
print("ALL LEAKAGE CHECKS PASSED.")

ALL LEAKAGE CHECKS PASSED.


## 11 — Final Feature Matrix Assembly

One clean file: `id` columns + `FEATURE_COLS` + `high_crime_day` +
`split`. Same-day crime columns and the fitted threshold are **not**
included — modeling only needs the target, not its derivation inputs.

In [25]:
FINAL_COLS = ["community_area", "date", "split"] + FEATURE_COLS + ["high_crime_day"]
modeling_dataset = df_clean[FINAL_COLS].copy()
print(f"final modeling dataset: {modeling_dataset.shape}")
modeling_dataset.head()

final modeling dataset: (138446, 32)


,community_area,date,split,year,month,day_of_week,week_of_year,quarter,season,is_weekend,avg_temp,temp_range,PRCP,SNOW,AWND,is_rain,is_snow,is_heavy_rain,is_heavy_snow,is_extreme_heat,is_extreme_cold,crime_count_lag_1,crime_count_lag_3,crime_count_lag_7,crime_count_lag_14,crime_count_lag_28,crime_rolling_7,crime_rolling_14,crime_rolling_28,recent_7d_avg,previous_7d_avg,high_crime_day
0,1,2021-01-29,train,2021,1,4,4,1,Winter,False,24.0,16,0.00,0.0,6.04,False,False,False,False,False,True,11.0,1.0,3.0,9.0,16.0,41.0,113.0,236.0,5.857143,10.285714,0
1,1,2021-01-30,train,2021,1,5,4,1,Winter,True,31.5,5,0.36,4.5,16.55,True,True,False,True,False,False,8.0,6.0,10.0,11.0,6.0,46.0,112.0,228.0,6.571429,9.428571,0
2,1,2021-01-31,train,2021,1,6,4,1,Winter,True,31.0,2,0.29,6.3,16.78,True,True,False,True,False,False,9.0,11.0,7.0,10.0,10.0,45.0,110.0,231.0,6.428571,9.285714,0
3,1,2021-02-01,train,2021,2,0,5,1,Winter,False,27.5,11,0.00,0.0,11.18,False,False,False,False,False,False,3.0,8.0,3.0,10.0,7.0,41.0,103.0,224.0,5.857143,8.857143,0
4,1,2021-02-02,train,2021,2,1,5,1,Winter,False,27.0,12,0.00,0.0,9.17,False,False,False,False,False,False,3.0,9.0,1.0,15.0,11.0,41.0,96.0,220.0,5.857143,7.857143,0


In [26]:
assert not excluded_all & set(modeling_dataset.columns), \
    "an excluded (leaky / redundant / structural-NA-risk) column leaked into final output -- stop"
assert modeling_dataset.isna().sum().sum() == 0
assert modeling_dataset["split"].isin(["train", "validation", "test"]).all()
print("Final assembly checks passed.")

Final assembly checks passed.


In [27]:
feature_metadata_rows = []
for col in FEATURE_COLS:
    if col in CALENDAR_FEATURE_COLS:
        source = "calendar (features.temporal_features)"
    elif col in WEATHER_FEATURE_COLS:
        source = "weather (features.weather_features, via 06 merge)"
    else:
        source = "historical crime lag/rolling (features.crime_lags)"
    feature_metadata_rows.append({
        "column": col, "dtype": str(modeling_dataset[col].dtype), "source": source,
        "leakage_note": "prior-day only, verified in Section 05" if col in HISTORICAL_FEATURE_COLS
                        else "same-day, non-crime-outcome -- safe" ,
    })
feature_metadata_rows.append({
    "column": "high_crime_day", "dtype": str(modeling_dataset["high_crime_day"].dtype),
    "source": f"derived: crime_count > training-only {int(TARGET_PERCENTILE*100)}th pctile per area",
    "leakage_note": "threshold fit on train years only; applied unchanged to val/test",
})
feature_metadata_df = pd.DataFrame(feature_metadata_rows)
feature_metadata_df

,column,dtype,source,leakage_note
0,year,int32,calendar (features.temporal_features),"same-day, non-crime-outcome -- safe"
1,month,int32,calendar (features.temporal_features),"same-day, non-crime-outcome -- safe"
2,day_of_week,int32,calendar (features.temporal_features),"same-day, non-crime-outcome -- safe"
3,week_of_year,int64,calendar (features.temporal_features),"same-day, non-crime-outcome -- safe"
4,quarter,int32,calendar (features.temporal_features),"same-day, non-crime-outcome -- safe"
5,season,object,calendar (features.temporal_features),"same-day, non-crime-outcome -- safe"
6,is_weekend,bool,calendar (features.temporal_features),"same-day, non-crime-outcome -- safe"
7,avg_temp,float64,"weather (features.weather_features, via 06 merge)","same-day, non-crime-outcome -- safe"
8,temp_range,int64,"weather (features.weather_features, via 06 merge)","same-day, non-crime-outcome -- safe"
9,PRCP,float64,"weather (features.weather_features, via 06 merge)","same-day, non-crime-outcome -- safe"


## 12 — Save & Verify

In [28]:
modeling_dataset.to_parquet(OUT_DATASET_PATH, index=False)
feature_metadata_df.to_csv(OUT_METADATA_PATH, index=False)
print(f"saved {len(modeling_dataset)} rows x {modeling_dataset.shape[1]} cols -> {OUT_DATASET_PATH}")
print(f"saved feature metadata -> {OUT_METADATA_PATH}")

saved 138446 rows x 32 cols -> C:\Users\Tarankit\Documents\Code\spatial-temporal-crime-trajectory-analytics\data\processed\modeling_dataset.parquet
saved feature metadata -> C:\Users\Tarankit\Documents\Code\spatial-temporal-crime-trajectory-analytics\data\processed\feature_metadata.csv


In [29]:
reloaded = pd.read_parquet(OUT_DATASET_PATH)
assert reloaded.shape == modeling_dataset.shape
assert reloaded["split"].value_counts().to_dict() == modeling_dataset["split"].value_counts().to_dict()
assert reloaded.isna().sum().sum() == 0
print("Round-trip verified.")

Round-trip verified.


## 13 — Findings

### Dataset
- Final shape: **138,446 rows × 32 columns** (3 id/split + 28 features +
  target), from 140,602 loaded — 1.53% dropped, entirely 2021 warm-up
  (2,156 rows, exactly 28 days × 77 areas).
- 28 modeling features: 7 calendar, 11 weather, 10 historical-crime
  (lag/rolling; `trend_ratio`/`violent_ratio`/`property_ratio` excluded).

### Target (`high_crime_day`)
- 77 area-level thresholds from training data only: mean 9.93, median 7,
  range 1–38 — real heterogeneity, not a single citywide cutoff.
- Overall rate 25.8%, but **drifts by year**: 15.7% (2021) → 25.2% (2022)
  → 32.4% (2023) → 31.1% (2024) → 24.5% (2025).
- By split: train 20.7%, validation 32.4%, test 27.8% — validation is the
  hardest split (Section 09).

### Leakage Audit
- **All 13 checks passed** (Section 10) — target fit on train years only,
  no same-day/redundant/structural-NA-risk column in the feature set,
  lag_1 and rolling_7 independently verified against a true shift/sum
  reconstruction, warm-up drop confirmed 2021-only, split strictly
  chronological and matching `config.py` exactly.

### Design decision worth flagging forward
- `trend_ratio`/`violent_ratio`/`property_ratio` were dropped from
  features (Section 06) — NaN in all 5 years (1,119 / 580 / 580 rows),
  not just warm-up, from `crime_lags.py`'s divide-by-zero guard. Keeping
  them would have removed validation/test rows non-randomly. If these
  ratios are wanted later, they need a documented zero-handling rule
  (e.g. a "no prior activity" flag), not row-dropping.

**Not performed here (by design):** model training, hyperparameter
tuning, imbalance handling (`class_weight`/SMOTE — Phase 15), evaluation
metrics.


---

## Overall Conclusion

`modeling_dataset.parquet` — **138,446 rows × 32 columns**, community_area
× date, 2021-2025 — passed all 13 leakage checks and is ready for
notebook 08.

Every predictor is verifiably safe: calendar and same-day weather are
non-crime-derived, and historical-crime features are **proven** prior-day
only (lag_1 checked against a true 1-day shift, rolling_7 checked against
an independent shift-then-sum reconstruction — not just named
correctly). Same-day crime outcome columns are present in the source data
but excluded from `X` by design; `TMAX`/`TMIN` excluded as redundant with
`avg_temp`; three ratio features excluded because their NaN pattern
(divide-by-zero guard in `crime_lags.py`) spans all 5 years, not just
warm-up, and would have biased which validation/test rows survived.

The headline finding is **temporal drift in the target itself**:
`high_crime_day` rate climbs from 15.7% (2021) to a peak of 32.4% (2023),
easing to 24.5% by 2025. Validation (2023) is the hardest split as a
direct result. This is not a data-quality problem — it is evidence that
Chicago crime levels shifted relative to the training-period baseline,
and notebook 08 should read validation/test performance with that context,
not assume a stationary class balance.

**Handed to notebook 08:** `modeling_dataset.parquet` (28 features +
target + split column) and `feature_metadata.csv`. No further
leakage-relevant decisions remain open — 08 should load, filter by
`split`, and go straight to baseline + model training.
